In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown


def detect_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing a results directory.")


project_root = detect_project_root()
results_root = project_root / "results"
metrics_root = results_root / "metrics"
tables_root = results_root / "tables"
tables_root.mkdir(parents=True, exist_ok=True)


def load_csv(*paths):
    for path in paths:
        if path.exists():
            return pd.read_csv(path)
    print("Skipping missing file(s): " + ", ".join(str(path) for path in paths))
    return pd.DataFrame()


def normalise_results(df):
    if df.empty:
        return df

    df = df.copy()
    df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed")]
    df.columns = [str(c).strip() for c in df.columns]

    df = df.rename(columns={
        "Dataset": "dataset",
        "Model": "model",
        "Accuracy": "accuracy",
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "FPR": "false_positive_rate",
        "Inference_ms": "inference_time_ms_per_email",
        "inference_time_ms_per_email": "inference_time_ms_per_email",
        "inference_time_per_email_sec": "inference_time_sec_per_email",
    })

    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].astype(str).str.strip()

    if "model" in df.columns:
        df["model"] = df["model"].astype(str).str.strip()

    if "inference_time_ms_per_email" not in df.columns and "inference_time_sec_per_email" in df.columns:
        df["inference_time_ms_per_email"] = pd.to_numeric(
            df["inference_time_sec_per_email"], errors="coerce"
        ) * 1000

    if "model" in df.columns and "Config" in df.columns:
        simhash_mask = df["model"].str.contains("SimHash", case=False, na=False)
        df.loc[simhash_mask, "model"] = df.loc[simhash_mask].apply(
            lambda row: f"{row['model']} ({row['Config']})" if pd.notna(row["Config"]) else row["model"],
            axis=1,
        )

    keep_cols = [
        "dataset",
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "false_positive_rate",
        "inference_time_ms_per_email",
    ]

    for col in keep_cols:
        if col not in df.columns:
            df[col] = pd.NA

    return df[keep_cols]


frames = [
    normalise_results(load_csv(metrics_root / "kaggle_classical_baseline_results.csv")),
    normalise_results(load_csv(metrics_root / "kaggle_dl_results.csv", results_root / "kaggle_dl_results.csv")),
    normalise_results(load_csv(metrics_root / "kaggle_distilbert_results.csv")),
    normalise_results(load_csv(metrics_root / "meajor_classical_baselines_results.csv")),
    normalise_results(load_csv(metrics_root / "meajor_dl_baseline_results.csv")),
    normalise_results(load_csv(metrics_root / "meajor_distilbert_results.csv")),
    normalise_results(load_csv(metrics_root / "all_results.csv", results_root / "all_results.csv")),
]

available_frames = [df for df in frames if not df.empty]
if not available_frames:
    raise FileNotFoundError(f"No result CSVs found under {metrics_root}.")

all_results = pd.concat(available_frames, ignore_index=True)

model_order = {
    "TF-IDF + MultinomialNB": 1,
    "TF-IDF + LogisticRegression": 2,
    "TF-IDF + LinearSVM": 3,
    "TF-IDF + RandomForest": 4,
    "SimHash + kNN (128-bit, k=1)": 5,
    "SimHash + kNN (128-bit, k=9)": 5,
    "TextCNN": 6,
    "BiLSTM": 7,
    "DistilBERT": 8,
}

pretty_names = {
    "TF-IDF + MultinomialNB": "TF-IDF\nMultinomial NB",
    "TF-IDF + LogisticRegression": "TF-IDF\nLogistic Regression",
    "TF-IDF + LinearSVM": "TF-IDF\nLinearSVM",
    "TF-IDF + RandomForest": "TF-IDF Random\nForest",
    "SimHash + kNN (128-bit, k=1)": "SimHash kNN\n(128-bit, k=1)",
    "SimHash + kNN (128-bit, k=9)": "SimHash kNN\n(128-bit, k=9)",
    "TextCNN": "TextCNN",
    "BiLSTM": "BiLSTM",
    "DistilBERT": "DistilBERT",
}


def build_table(dataset_name):
    df = all_results[all_results["dataset"].str.lower() == dataset_name.lower()].copy()
    df["sort_order"] = df["model"].map(model_order).fillna(999)
    df = df.sort_values(["sort_order", "model"]).drop(columns="sort_order")

    df["model"] = df["model"].map(lambda x: pretty_names.get(x, x))

    df = df.rename(columns={
        "model": "Model",
        "accuracy": "Accuracy",
        "precision": "Precision",
        "recall": "Recall",
        "f1": "F1",
        "false_positive_rate": "FPR",
        "inference_time_ms_per_email": "Inference\n(ms/email)",
    })

    return df[[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "FPR",
        "Inference\n(ms/email)",
    ]]


kaggle_table = build_table("Kaggle")
meajor_table = build_table("MeAJOR")

kaggle_table.to_csv(tables_root / "kaggle_summary_table.csv", index=False)
meajor_table.to_csv(tables_root / "meajor_summary_table.csv", index=False)
all_results.to_csv(tables_root / "all_results_normalised.csv", index=False)

display(Markdown("## Table: Kaggle dataset"))
display(
    kaggle_table.style
    .hide(axis="index")
    .format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "FPR": "{:.4f}",
        "Inference\n(ms/email)": "{:.4f}",
    })
)

display(Markdown("## Table: MeAJOR dataset"))
display(
    meajor_table.style
    .hide(axis="index")
    .format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "FPR": "{:.4f}",
        "Inference\n(ms/email)": "{:.4f}",
    })
)

print(f"Saved CSV tables to: {tables_root}")

## Table: Kaggle dataset

Model,Accuracy,Precision,Recall,F1,FPR,Inference (ms/email)
TF-IDF Multinomial NB,0.8631,1.0000,0.5094,0.6749,0.0000,0.0013
TF-IDF Logistic Regression,0.9415,1.0000,0.7903,0.8828,0.0000,0.0014
TF-IDF LinearSVM,0.9906,0.9962,0.9700,0.9829,0.0014,0.0004
TF-IDF Random Forest,0.9833,0.9960,0.9438,0.9692,0.0014,0.0278
"SimHash kNN (128-bit, k=1)",0.9122,0.8828,0.7903,0.8340,0.0406,0.0504
TextCNN,0.9833,0.9846,0.9551,0.9696,0.0058,1.5106
BiLSTM,0.9666,0.9181,0.9663,0.9416,0.0333,1.1693
DistilBERT,0.9885,0.9923,0.9663,0.9791,0.0029,23.6200


## Table: MeAJOR dataset

Model,Accuracy,Precision,Recall,F1,FPR,Inference (ms/email)
TF-IDF Multinomial NB,0.9630,0.9846,0.9316,0.9574,0.0117,0.0006
TF-IDF Logistic Regression,0.9770,0.9759,0.9724,0.9741,0.0193,0.0002
TF-IDF LinearSVM,0.9845,0.9819,0.9834,0.9826,0.0146,0.0002
TF-IDF Random Forest,0.9775,0.9857,0.9634,0.9744,0.0112,0.0378
"SimHash kNN (128-bit, k=9)",0.8615,0.8493,0.8379,0.8436,0.1196,1.2267
TextCNN,0.9831,0.9768,0.9855,0.9811,0.0189,0.3211
BiLSTM,0.9792,0.9779,0.9753,0.9766,0.0177,3.1347
DistilBERT,0.9896,0.9923,0.9842,0.9882,0.0062,7.6370


Saved CSV tables to: /Users/ziliang/Downloads/cm3203-phishing-nlp/results/tables
